# Recover Missing Artifacts — Weighted Loss Run

The original `finetune_weighted_loss.ipynb` run used `mlflow.log_artifacts()`
(raw file copy) instead of `mlflow.transformers.log_model()`, producing only:
- `config.json`, `model.safetensors`, `training_args.bin`

**Missing vs the unfrozen run's full artifact set:**
- `MLmodel`, `conda.yaml`, `python_env.yaml`, `requirements.txt`
- `components/`, `image_processor/`, `metadata/`
- `id2label` / `label2id` in `config.json`

This notebook fixes everything by:
1. Rebuilding class names directly from the ontology JSON (no project wheel needed)
2. Patching `id2label`/`label2id` into `config.json`
3. Loading the model + image processor and logging via `mlflow.transformers.log_model()`

**No project wheel install required** — all ontology logic is inlined.
**GPU required** — `UperNetForSemanticSegmentation.from_pretrained()` loads weights.

In [0]:
# Cell 0 — Configuration

import json
import os
import time

WORKSPACE_BASE = "/Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology"
ONTOLOGY_PATH = f"{WORKSPACE_BASE}/ontology/structure_graph_1.json"
FINAL_MODEL_DIR = "/dbfs/FileStore/allen_brain_data/models/weighted-loss"

HF_ENDPOINT = os.environ.get(
    "HF_ENDPOINT",
    "https://graingerreadonly.jfrog.io/artifactory/api/huggingfaceml/huggingfaceml-remote",
)
MODEL_REPO_ID = "facebook/dinov2-large"
MODEL_CACHE_DIR = "/tmp/dinov2-large"

EXPERIMENT_NAME = "/Users/noel.nosse@grainger.com/histology-brain-segmentation"
REGISTERED_MODEL_NAME = "dinov2-upernet-weighted-loss"

print(f"Ontology:    {ONTOLOGY_PATH}")
print(f"Model dir:   {FINAL_MODEL_DIR}")
print(f"HF endpoint: {HF_ENDPOINT}")

Ontology:    /Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology/ontology/structure_graph_1.json
Model dir:   /dbfs/FileStore/allen_brain_data/models/weighted-loss
HF endpoint: https://graingerreadonly.jfrog.io/artifactory/api/huggingfaceml/huggingfaceml-remote


In [0]:
# Cell 1 — Rebuild class names from ontology JSON (no wheel needed)
#
# Replicates OntologyMapper.build_full_mapping() + get_class_names() inline.

def _flatten_ontology(node, depth=0):
    """Recursively flatten the ontology tree into a list of nodes."""
    result = [{k: v for k, v in node.items() if k != 'children'}]
    result[-1]['depth'] = depth
    for child in node.get('children', []):
        result.extend(_flatten_ontology(child, depth + 1))
    return result


def build_class_names(ontology_path):
    """Load ontology, build full mapping, return (num_labels, class_names)."""
    with open(ontology_path) as f:
        data = json.load(f)

    nodes = _flatten_ontology(data['msg'][0])
    node_by_id = {n['id']: n for n in nodes}
    all_structure_ids = set(node_by_id.keys())

    # build_full_mapping: sorted structure IDs -> contiguous class IDs
    sorted_ids = sorted(all_structure_ids)
    mapping = {sid: i + 1 for i, sid in enumerate(sorted_ids)}

    num_labels = max(mapping.values()) + 1  # includes class 0 (background)

    # get_class_names: for each class_id, pick the shallowest-depth structure
    names = ['Background'] * num_labels
    class_to_sid = {}
    for sid, cid in mapping.items():
        if cid == 0:
            continue
        current = class_to_sid.get(cid)
        if current is None:
            class_to_sid[cid] = sid
        elif node_by_id[sid]['depth'] < node_by_id[current]['depth']:
            class_to_sid[cid] = sid

    for cid, sid in class_to_sid.items():
        node = node_by_id.get(sid)
        if node is not None:
            names[cid] = node['name']

    return num_labels, names


num_labels, class_names = build_class_names(ONTOLOGY_PATH)
print(f"Ontology loaded: {num_labels} classes")
print(f"Sample: class 0 = {class_names[0]}, class 1 = {class_names[1]}")

Ontology loaded: 1328 classes
Sample: class 0 = Background, class 1 = Tuberomammillary nucleus, ventral part


In [0]:
# Cell 2 — Patch config.json with id2label / label2id

config_path = os.path.join(FINAL_MODEL_DIR, 'config.json')
with open(config_path) as f:
    config = json.load(f)

config['id2label'] = {str(i): name for i, name in enumerate(class_names)}
config['label2id'] = {name: str(i) for i, name in enumerate(class_names)}

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"Patched config.json with id2label/label2id ({len(class_names)} entries)")

Patched config.json with id2label/label2id (1328 entries)


In [0]:
# Cell 3 — Download base model image processor + save to model dir

from huggingface_hub import snapshot_download
from transformers import AutoImageProcessor

print(f"Downloading {MODEL_REPO_ID} from Artifactory...")
for attempt in range(1, 4):
    try:
        base_model_path = snapshot_download(
            repo_id=MODEL_REPO_ID,
            endpoint=HF_ENDPOINT,
            etag_timeout=86400,
            cache_dir=MODEL_CACHE_DIR,
        )
        break
    except Exception as e:
        if attempt < 3:
            wait = 2 ** attempt
            print(f"  Attempt {attempt} failed ({e.__class__.__name__}), retrying in {wait}s...")
            time.sleep(wait)
        else:
            raise

processor = AutoImageProcessor.from_pretrained(base_model_path)
processor.save_pretrained(FINAL_MODEL_DIR)
print(f"Saved preprocessor_config.json to {FINAL_MODEL_DIR}")

# Verify all expected files
print(f"\nModel directory contents:")
for fname in sorted(os.listdir(FINAL_MODEL_DIR)):
    fpath = os.path.join(FINAL_MODEL_DIR, fname)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {fname:35s} {size_mb:>8.2f} MB")

2026-03-14 13:51:08.415454: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773496268.431138    6253 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773496268.435962    6253 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773496268.449333    6253 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773496268.449345    6253 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773496268.449347    6253 computation_placer.cc:177] computation placer alr

[2026-03-14 13:51:10,598] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Saved preprocessor_config.json to /dbfs/FileStore/allen_brain_data/models/weighted-loss

Model directory contents:
  config.json                             0.12 MB
  model.safetensors                    1305.14 MB
  preprocessor_config.json                0.00 MB
  training_args.bin                       0.01 MB


In [0]:
# Cell 4 — Load model + processor from patched DBFS dir

from transformers import UperNetForSemanticSegmentation, AutoImageProcessor

print("Loading model from DBFS (requires GPU memory)...")
model = UperNetForSemanticSegmentation.from_pretrained(FINAL_MODEL_DIR)
processor = AutoImageProcessor.from_pretrained(FINAL_MODEL_DIR)

num_params = sum(p.numel() for p in model.parameters())
print(f"Loaded model: {num_params:,} params, {model.config.num_labels} labels")
print(f"id2label present: {len(model.config.id2label)} entries")
print(f"Processor: {processor.__class__.__name__}")

Loading model from DBFS (requires GPU memory)...
Loaded model: 342,104,160 params, 1328 labels
id2label present: 1328 entries
Processor: BitImageProcessor


In [0]:
# Cell 5 — Log model to MLflow with full artifact packaging
#
# Uses mlflow.transformers.log_model() to produce the complete artifact set:
#   MLmodel, conda.yaml, python_env.yaml, requirements.txt,
#   components/, image_processor/, metadata/, model/
#
# NOTE: registered_model_name requires a 3-level Unity Catalog path
# (catalog.schema.model). Register separately via the UC UI or:
#   import mlflow
#   mlflow.register_model(
#       f"runs:/{run_id}/model",
#       "your_catalog.your_schema.dinov2-upernet-weighted-loss",
#   )

import mlflow
import mlflow.transformers

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName LIKE 'weighted-loss-%'",
    order_by=["start_time DESC"],
    max_results=1,
)
if len(runs) == 0:
    raise RuntimeError("Could not find original weighted-loss MLflow run")

run_id = runs.iloc[0].run_id
run_name = runs.iloc[0].get('tags.mlflow.runName', 'N/A')
print(f"Found MLflow run: {run_name} ({run_id})")
print(f"Logging model — this will take several minutes for a 1.3 GB model...\n")

with mlflow.start_run(run_id=run_id):
    mlflow.transformers.log_model(
        transformers_model={
            "model": model,
            "image_processor": processor,
        },
        name="model",
        task="image-segmentation",
        metadata={
            "source": "recovery_from_incomplete_save",
            "original_path": FINAL_MODEL_DIR,
            "num_labels": model.config.num_labels,
            "mapping_type": "full",
            "loss_type": "combined_dice_ce",
        },
    )
    mlflow.set_tag("dbfs_model_path", FINAL_MODEL_DIR)

print(f"\nModel logged to MLflow run: {run_id}")
print(f"Load with: mlflow.transformers.load_model('runs:/{run_id}/model')")

Found MLflow run: weighted-loss-1328class-20260312-2204 (55e3e0c9c98a4c65ade76bd65f2245ee)
Logging model — this will take several minutes for a 1.3 GB model...



🔗 View Logged Model at: https://grainger-gtg-mlops-dev-use1-dbx-shared-ws.cloud.databricks.com/ml/experiments/1345391216675532/models/m-95cd24ee43794ec399abe0ca4ead339a?o=2648165776397546
Device set to use cuda:0
2026/03/14 13:52:22 WARNING mlflow.transformers: The model card could not be retrieved from the hub due to Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/dbfs/FileStore/allen_brain_data/models/weighted-loss'. Use `repo_type` argument if needed.
2026/03/14 13:52:22 WARNING mlflow.transformers: Unable to find license information for this model. Please verify permissible usage for the model you are storing prior to use.
2026/03/14 13:52:22 WARNING mlflow.transformers: This model is unable to be used for pyfunc prediction because the model is not a language-based model and requires a complex input type that is currently not supported. The pyfunc flavor will not be added to the Model.
2026/03/14 13:52:29 WARNING mlflow.utils.environment: Encountered an unexpect


Model logged to MLflow run: 55e3e0c9c98a4c65ade76bd65f2245ee
Load with: mlflow.transformers.load_model('runs:/55e3e0c9c98a4c65ade76bd65f2245ee/model')


In [0]:
# Cell 6 — Verify artifact tree

artifacts = mlflow.artifacts.list_artifacts(run_id=run_id)
print(f"Artifacts in run {run_id}:")
for artifact in artifacts:
    print(f"  {artifact.path}")
    if artifact.is_dir:
        sub_artifacts = mlflow.artifacts.list_artifacts(
            run_id=run_id, artifact_path=artifact.path,
        )
        for sub in sub_artifacts:
            print(f"    {sub.path}")

print(f"\n--- Recovery complete ---")

Artifacts in run 55e3e0c9c98a4c65ade76bd65f2245ee:
  model
    model/config.json
    model/model.safetensors
    model/preprocessor_config.json
    model/training_args.bin
    model/LICENSE.txt
    model/MLmodel
    model/components
    model/conda.yaml
    model/metadata
    model/model
    model/python_env.yaml
    model/requirements.txt

--- Recovery complete ---
